# Topología Aplicada y Computacional

## Práctica 1

**Ejercicio 1:** Definir una clase para almacenar complejos simpliciales.

Antes de crear la clase `SimplicialComplex` se desarrolla la función `power_set`, que calcula el conjunto de las partes de una tupla. La implementación aprovecha la representación en base $2$ de los números para codificar cada conjunto del conjunto potencia. De esta manera, el elemento $i$-ésimo de la tupla se incluye en cada iteración si el bit $i$-ésimo del número es $1$. Recorriendo todos los números hasta $2^N - 1$, dónde $N$ es la longitud de la tupla, se obtiene el resultado.

In [35]:
def power_set(simplicial: tuple[int, ...]) -> list[tuple[int, ...]]:
    n = len(simplicial)

    res = []

    for i in range(1, 1 << n):
        res.append(tuple(
            simplicial[j]for j in range(n) if i & (1 << j)
        ))

    return res

A continuación se implementa la clase. Para crear un objeto `SimplicialComplex` basta con proveer el iterable `generator` con los símplices iniciales, sin que haga falta añadir todas sus caras (ya que esto se hace automáticamente computando su conjunto potencia). Por otro lado, la cardinalidad se obtiene del símplice generador de mayor cardinalidad.

Se ha decidido almacenar como atributo todos los símplices del complejo simplicial, en lugar de sólo los maximales, para no tener que recomputarlos cada vez que se quiera consultar el `face_set`. De no ser así, cada llamada supondría una operación de complejidad $O(2^N)$.

In [36]:
class SimplicialComplex:
    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

En efecto, aunque el complejo simplicial sólo se defina con el triángulo $\{1,2,3\}$, las caras restantes se añaden automáticamente.

In [37]:
SimplicialComplex([(1,2,3)]).face_set

{(1,), (1, 2), (1, 2, 3), (1, 3), (2,), (2, 3), (3,)}

**Ejercicio 2:** Definir una función que devuelva la dimensión del complejo simplicial.

Como ya se guarda la cardinalidad como atributo, basta con restarle uno para obtener la dimensión.

In [38]:
class SimplicialComplex:
    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

De nuevo, se comprueba su funcionamiento.

In [39]:
SimplicialComplex([(1,2,3)]).dimension

2

**Ejercicio 3:** Definir una función que devuelva el conjunto de todas las caras del complejo simplicial.

Como en el constructor ya computa el conjunto de caras y lo guarda en el atributo `face_set`, el trabajo ya está hecho.

**Ejercicio 4:** Definir una función que devuelva el conjunto de todas las caras de dimensión dada.

Basta con filtrar las caras del atributo `face_set` con la dimensión indicada.

In [40]:
class SimplicialComplex:
    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

    def n_faces(self, dim: int) -> set[tuple[int, ...]]:
        return {face for face in self.face_set if len(face) - 1 == dim}

Comprobamos su funcionamiento.

In [41]:
SimplicialComplex([(1,2,3,4)]).n_faces(1)

{(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)}

**Ejercicio 5:** Definir una función que calcule la estrella de un símplice.

Para comenzar, se desarrolla la función auxiliar `is_face`, que indica si el símplice `s1` es cara de `s2` utilizando la definición de cara de un símplice abstracto. A partir de esta función se implementa `st` que computa la estrella del parámetro usando la definición y emite un error en caso de que el símplice provisto no sea una cara del complejo simplicial.

In [42]:
class SimplicialComplex:
    @staticmethod
    def is_face(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i in s2 for i in s1)

    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

    def n_faces(self, dim: int) -> set[tuple[int, ...]]:
        return {face for face in self.face_set if len(face) - 1 == dim}

    def st(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        res = {member for member in self.face_set if self.is_face(simplicial, member)}

        if not res:
            raise ValueError("'simplicial' is not a face")

        return res

Finalmente, se comprueba el funcionamiento del método.

In [43]:
SimplicialComplex([(1,2,3)]).st((1,))

{(1,), (1, 2), (1, 2, 3), (1, 3)}

**Ejercicio 6:** Definir una función que calcule el link de un símplice.

El link de un símplice $\tau$ es el conjunto de símplices cuya intersección con la estrella cerrada de $\tau$ es vacía. De este modo para implementar el método `link` es primero necesario un método para computar la estrella cerrada de un símplice. Esto es sencillo, ya que basta con usar el constructor de `SimplicialComplex` con la estrella abierta como generador, produciendo así el mínimo complejo simplicial que contiene la estrella. De este modo, una vez implementado `closed_st` basta con usar la definición de link y la función auxiliar `is_disjoint` para implementar el método `lk`.

In [44]:
class SimplicialComplex:
    @staticmethod
    def is_face(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i in s2 for i in s1)

    @staticmethod
    def is_disjoint(s1: tuple[int, ...], s2: tuple[int, ...]) -> bool:
        return all(i not in s1 for i in s2)

    def __init__(self, generator: Iterable[tuple[int, ...]]) -> None:
        self.cardinality = 0
        self.face_set = set() 

        for simplicial in generator:
            self.face_set.update(power_set(simplicial))
            if len(simplicial) > self.cardinality:
                self.cardinality = len(simplicial)

    @property
    def dimension(self) -> int:
        return self.cardinality - 1

    def n_faces(self, dim: int) -> set[tuple[int, ...]]:
        return {face for face in self.face_set if len(face) - 1 == dim}

    def st(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        res = {member for member in self.face_set if self.is_face(simplicial, member)}

        if not res:
            raise ValueError("'simplicial' is not a face")

        return res

    def closed_st(self, simplicial: tuple[int, ...]) -> SimplicialComplex:
        return SimplicialComplex(self.st(simplicial))

    def lk(self, simplicial: tuple[int, ...]) -> set[tuple[int, ...]]:
        closed = self.closed_st(simplicial)
        return {elem for elem in closed.face_set if self.is_disjoint(simplicial, elem)}

Se comprueba que su funcionamiento es correcto.

In [34]:
SimplicialComplex([(1,2,3)]).lk((1,))

{(2,), (2, 3), (3,)}